# Training a compact Vision Transformer

Run an offline experiment on synthetic image patterns. Accuracy here checks the training pipeline; it is not a real-dataset result.

In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / "edgeformer").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "edgeformer").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks folder")
sys.path.insert(0, str(ROOT))
torch.set_num_threads(1)
torch.manual_seed(42)
from edgeformer.models import TinyViT

model = TinyViT(
    image_size=16,
    patch_size=4,
    num_classes=4,
    embed_dim=32,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.0,
)

In [ ]:
from edgeformer.utils.dataset import get_loaders
from edgeformer.engine import train_epoch, evaluate

train, val = get_loaders(
    "synthetic", image_size=16, num_classes=4, batch_size=16, train_samples=64, val_samples=32
)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
history = []
for epoch in range(3):
    train_metrics = train_epoch(model, train, optimizer)
    val_metrics = evaluate(model, val)
    history.append({"epoch": epoch + 1, "train": train_metrics, "val": val_metrics})
history

In [ ]:
import matplotlib.pyplot as plt

plt.plot([r["epoch"] for r in history], [r["train"]["loss"] for r in history], "o-", label="train")
plt.plot(
    [r["epoch"] for r in history], [r["val"]["loss"] for r in history], "o-", label="validation"
)
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.legend()
plt.grid(alpha=0.2)

For resumable training with best/latest checkpoints and a cosine schedule, use `python -m edgeformer train --config configs/smoke.yaml --output runs/smoke` from a terminal.